In [1]:
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

# Load and combine all dataset parts (sorted for reproducibility)
files = sorted(glob.glob("../data/raw/uwb_dataset_part*.csv"))
print("Files found:", len(files))
print(files)

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Dataset shape:", df.shape)
df.head()

Files found: 7
['../data/raw\\uwb_dataset_part1.csv', '../data/raw\\uwb_dataset_part2.csv', '../data/raw\\uwb_dataset_part3.csv', '../data/raw\\uwb_dataset_part4.csv', '../data/raw\\uwb_dataset_part5.csv', '../data/raw\\uwb_dataset_part6.csv', '../data/raw\\uwb_dataset_part7.csv']
Dataset shape: (42000, 1031)


,NLOS,RANGE,FP_IDX,FP_AMP1,FP_AMP2,FP_AMP3,STDEV_NOISE,CIR_PWR,MAX_NOISE,RXPACC,...,CIR1006,CIR1007,CIR1008,CIR1009,CIR1010,CIR1011,CIR1012,CIR1013,CIR1014,CIR1015
0,0.0,3.90,745.0,18712.0,10250.0,11576.0,64.0,11855.0,967.0,611.0,...,279.0,458.0,183.0,158.0,198.0,87.0,296.0,505.0,307.0,0.0
1,0.0,0.66,749.0,11239.0,6313.0,4712.0,64.0,18968.0,1133.0,447.0,...,144.0,334.0,290.0,228.0,187.0,213.0,202.0,89.0,103.0,0.0
2,1.0,7.86,746.0,4355.0,5240.0,3478.0,60.0,14699.0,894.0,723.0,...,32.0,373.0,224.0,174.0,124.0,329.0,207.0,96.0,218.0,0.0
3,1.0,3.48,750.0,8502.0,8416.0,5890.0,76.0,8748.0,1127.0,1024.0,...,252.0,173.0,198.0,160.0,434.0,397.0,290.0,155.0,342.0,256.0
4,0.0,1.19,746.0,17845.0,18095.0,12058.0,68.0,11380.0,1744.0,276.0,...,154.0,209.0,242.0,296.0,87.0,178.0,314.0,247.0,292.0,256.0


In [2]:
CLASS_COL = "NLOS"
RANGE_COL = "RANGE"

y_class = df[CLASS_COL]
y_range = df[RANGE_COL]

print("y_class shape:", y_class.shape)
print("y_range shape:", y_range.shape)

y_class shape: (42000,)
y_range shape: (42000,)


In [3]:
all_idx = df.index.to_numpy()

train_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.2,
    random_state=42,
    stratify=y_class,   # stratify by class
    shuffle=True
)

print("Train size:", len(train_idx))
print("Test size:", len(test_idx))

Train size: 33600
Test size: 8400


In [4]:
output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

# Extract Y using indices
y_train_class = y_class.loc[train_idx].reset_index(drop=True)
y_test_class  = y_class.loc[test_idx].reset_index(drop=True)

y_train_range = y_range.loc[train_idx].reset_index(drop=True)
y_test_range  = y_range.loc[test_idx].reset_index(drop=True)

# Save Y files required by your structure
y_train_class.to_csv(output_path / "y_train_class.csv", index=False)
y_test_class.to_csv(output_path / "y_test_class.csv", index=False)

y_train_range.to_csv(output_path / "y_train_range.csv", index=False)
y_test_range.to_csv(output_path / "y_test_range.csv", index=False)

# Save indices so all X variants match exactly
pd.Series(train_idx, name="idx").to_csv(output_path / "train_idx.csv", index=False)
pd.Series(test_idx, name="idx").to_csv(output_path / "test_idx.csv", index=False)

print("Saved into:", output_path)
print(" - y_train_class.csv, y_test_class.csv")
print(" - y_train_range.csv, y_test_range.csv")
print(" - train_idx.csv, test_idx.csv")

Saved into: ..\data\processed
 - y_train_class.csv, y_test_class.csv
 - y_train_range.csv, y_test_range.csv
 - train_idx.csv, test_idx.csv
